In [1]:
# Data is already extracted locally at src/Preprocessing/Output (created by resize_img.py)
from pathlib import Path
assert (Path.cwd() / "src" / "Preprocessing" / "Output").is_dir(), "Run this notebook from the repo root"

In [2]:
# src/Preprocessing/resize_img.py
import os
from pathlib import Path
from PIL import Image, ImageOps

# Paths are built from this file's location, so it works from any working directory
REPO_ROOT = Path.cwd()                               # notebook runs from repo root
BASE_DIR = REPO_ROOT / "src" / "Preprocessing"
input_dir = REPO_ROOT / "CarDD_release" / "CarDD_SOD"
output_dir = BASE_DIR / "Output"
TARGET_SIZE = (384, 576)
EXTENSIONS = (".png", ".jpg")
EXCLUDE_DIRS = {"CarDD-TE-Edge" ,"CarDD-TR-Edge",  "CarDD-VAL-Edge"}

def resize_image(image_path, output_folder, size):
    try:
        with Image.open(image_path) as img:
            # Masks/edge maps are usually single-channel, so pad with an int; colour images use a tuple
            color = 0 if img.mode in ("L", "1", "P") else (0, 0, 0)
            img = ImageOps.pad(img, size, method=Image.Resampling.LANCZOS, color=color)
            output_path = os.path.join(output_folder, os.path.basename(image_path))
            img.save(output_path)
    except Exception as e:
        print(f"Error resizing image {image_path}: {e}")

def create_new_output_dir(path):
    os.makedirs(path, exist_ok=True)

def get_output_directory(input_directory):
    relative_path = os.path.relpath(input_directory, input_dir)
    return os.path.join(output_dir, relative_path)

for subdir, dirs, files in os.walk(input_dir):
    dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
    for file in files:
        if file.lower().endswith(EXTENSIONS):
            input_file_path = os.path.join(subdir, file)
            output_directory = get_output_directory(subdir)
            create_new_output_dir(output_directory)
            output_file_path = os.path.join(output_directory, file)
            if not os.path.exists(output_file_path):
                resize_image(input_file_path, output_directory, TARGET_SIZE)


In [3]:
# src/unet_model.py
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNET(nn.Module):
    def __init__(
        self, in_channels=3, out_channels=1, features=(32, 64, 128, 256)
    ):
        super().__init__()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        for feature in reversed(features):
            self.ups.append(
                nn.ConvTranspose2d(
                    feature*2, feature, kernel_size=2, stride=2,
                )
            )
            self.ups.append(DoubleConv(feature*2, feature))

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)


    def forward(self, x):
        skip_connections = []

        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx//2]

            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](concat_skip)

        return self.final_conv(x)

#model = UNET()

#x = torch.randn(2, 3, 384, 576)

#y = model(x)

#print("Input:", x.shape)
#print("Output:", y.shape)


In [4]:
# src/training/BCEDiceLoss.py
import torch
import torch.nn as nn

class BCEDiceLoss(nn.Module):
    def __init__ (
            self, weight=0.5, eps=1e-7):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.weight = weight
        self.eps = eps

    def forward(self, logits, target):
        bce = self.bce(logits, target)

        probs = torch.sigmoid(logits)
        dims = (1, 2, 3)
        inter = (probs * target).sum(dims)
        union = probs.sum(dims) + target.sum(dims)
        dice_loss = 1 - ((2 * inter + self.eps) / (union + self.eps)).mean()

        return self.weight * bce + (1 - self.weight) * dice_loss


In [5]:
# src/training/metrics.py
import torch

def confusion(out, target, threshold=0.5):
    preds = (torch.sigmoid(out) > threshold).float()

    tp = (preds * target).sum()
    fp = (preds * (1 - target)).sum()
    fn = ((1 - preds) * target).sum()

    return tp, fp, fn

def scores(tp, fp, fn, eps=1e-7):                               # eps is epsilon, fixed errors dividing by zero
    return {
        "f1": ((2 * tp + eps) / (2 * tp + fp + fn + eps)).item(),
        "precision": ((tp + eps) / (tp + fp + eps)).item(),
        "recall": ((tp + eps) / (tp + fn + eps)).item(),
        "iou": ((tp + eps) / (tp + fp + fn + eps)).item()         # iou is intersection over union, the overlap between the predicted and true masks
    }


In [6]:
# src/training/plot.py
import matplotlib.pyplot as plt
import torch

def plot_history(history, save_path=None):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax_loss, ax_metric) = plt.subplots(1, 2, figsize=(12, 4))

    ax_loss.plot(epochs, history["train_loss"], label="train")
    ax_loss.plot(epochs, history["val_loss"], label="val")
    ax_loss.set_xlabel("epoch")
    ax_loss.set_ylabel("BCE+Dice loss")
    ax_loss.set_title("Loss")
    ax_loss.legend()

    for key in ("iou", "f1", "precision", "recall"):
        ax_metric.plot(epochs, history[key], label=key)
    ax_metric.set_xlabel("epoch")
    ax_metric.set_ylabel("score")
    ax_metric.set_title("Validation metrics")
    ax_metric.set_ylim(0, 1)
    ax_metric.legend()

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    else:
        plt.show()
    plt.close(fig)


@torch.no_grad()
def plot_predictions(model, loader, device, n=4, threshold=0.5, save_path=None):
    model.eval()
    images, masks = next(iter(loader))
    images, masks = images[:n], masks[:n]

    logits = model(images.to(device))
    preds = (torch.sigmoid(logits) > threshold).float().cpu()

    fig, axes = plt.subplots(n, 4, figsize=(12, 3 * n))
    if n == 1:
        axes = axes[None, :]

    col_titles = ["image", "ground truth", "prediction", "overlay"]
    for i in range(n):
        img = images[i].permute(1, 2, 0).numpy()
        gt = masks[i, 0].numpy()
        pr = preds[i, 0].numpy()

        axes[i, 0].imshow(img)
        axes[i, 1].imshow(gt, cmap="gray")
        axes[i, 2].imshow(pr, cmap="gray")
        axes[i, 3].imshow(img)
        axes[i, 3].imshow(pr, cmap="Reds", alpha=0.4)

        for j in range(4):
            axes[i, j].axis("off")
            if i == 0:
                axes[i, j].set_title(col_titles[j])

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    else:
        plt.show()
    plt.close(fig)


In [7]:
# src/training/train.py
import torch
import torch.nn as nn

import time
import statistics

from torch.utils.data import DataLoader

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    return running_loss / len(loader)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total, tp, fp, fn = 0, 0, 0, 0

    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        total += criterion(logits, masks).item()
        
        t, p, n = confusion(logits, masks)
        tp, fp, fn = tp + t, fp + p, fn + n
    return total / len(loader), scores(tp, fp, fn)

def train(train_loader, val_loader, epochs=30, save_path="unet_model.pth"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = UNET().to(device)
    criterion = BCEDiceLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    history = {"train_loss": [], "val_loss": [], "iou": [], "f1": [], "precision": [], "recall": []}
    best_iou = 0.0
    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_score = validate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        for k in ("iou", "f1", "precision", "recall"):
            history[k].append(val_score[k])

        print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Scores: {val_score}")

        if best_iou < val_score["iou"]:
            best_iou = val_score["iou"]
            torch.save(model.state_dict(), save_path)
            print(f"Model saved with IoU: {best_iou:.4f}")

    return model, history

def cross_validate(fold_loaders, epochs=10):
    fold_scores = []

    for i, (train_loader, val_loader) in enumerate(fold_loaders):
        print(f"\n===== Fold {i+1}/{len(fold_loaders)} =====")
        save_path = f"unet_fold{i+1}.pth"
        model, _ = train(train_loader, val_loader, epochs=epochs, save_path=save_path)

        model.load_state_dict(torch.load(save_path))
        device = next(model.parameters()).device
        _, best = validate(model, val_loader, BCEDiceLoss(), device)
        fold_scores.append(best)

    for key in ("iou", "f1", "precision", "recall"):
        values = [s[key] for s in fold_scores]
        print(f"{key}: {statistics.mean(values):.4f} +- {statistics.stdev(values):.4f}")

    return fold_scores


In [8]:
# src/Preprocessing/create_tensors.py
from pathlib import Path
import random
from torch.utils.data import Subset, ConcatDataset
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import sys
from sklearn.model_selection import KFold


BASE_DIR = Path.cwd() / "src" / "Preprocessing"     # notebook runs from repo root
input_dir = BASE_DIR / "Output"

class CarDDDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.names = sorted(p.stem for p in self.image_dir.iterdir()
                            if p.suffix.lower() in (".jpg", ".png"))
        
    def __len__(self):
        return len(self.names)
    
    def _find(self, folder, stem):
        for ext in (".jpg", ".png"):
            p = folder / (stem + ext)
            if p.exists():
                return p
        raise FileNotFoundError(f"{stem} not found in {folder}")
    
    def __getitem__(self, idx):
        name = self.names[idx]
        with Image.open(self._find(self.image_dir, name)) as f:
            img = f.convert("RGB")
        with Image.open(self._find(self.mask_dir, name)) as f:
            msk = f.convert("L")

        image = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        mask = (torch.from_numpy(np.array(msk)).unsqueeze(0) > 127).float()
        return image, mask
    

def split(name):
        return CarDDDataset(
            input_dir / f"CarDD-{name}" / f"CarDD-{name}-Image",
            input_dir / f"CarDD-{name}" / f"CarDD-{name}-Mask"
        )

def make_loaders(batch_size = 6):
    
    train_ds, val_ds, test_ds = split("TR"), split("VAL"), split("TE")
    return (
        DataLoader(train_ds, batch_size = batch_size, shuffle=True),
        DataLoader(val_ds, batch_size = batch_size, shuffle=False),
        DataLoader(test_ds, batch_size = batch_size, shuffle=False)
    )

def make_fold_loaders(k = 5, batch_size = 6):
    train_ds, val_ds = split("TR"), split("VAL")
    pool = ConcatDataset([train_ds, val_ds])

    kf = KFold(n_splits = k, shuffle=True, random_state=42)
    fold_loaders = []

    for train_idx, val_idx in kf.split(range(len(pool))):
        train_subset = Subset(pool, train_idx)
        val_subset = Subset(pool, val_idx)

        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
        fold_loaders.append((train_loader, val_loader))

    return fold_loaders


In [9]:
# src/test.py
if __name__ == "__main__":
    fold_loaders = make_fold_loaders(k=5)
    cross_validate(fold_loaders, epochs=10)



===== Fold 1/5 =====
Epoch [1/10], Train Loss: 0.5188, Val Loss: 0.4635, Val Scores: {'f1': 0.5693188309669495, 'precision': 0.48063409328460693, 'recall': 0.6981363296508789, 'iou': 0.39793556928634644}
Model saved with IoU: 0.3979
Epoch [2/10], Train Loss: 0.4524, Val Loss: 0.4182, Val Scores: {'f1': 0.608402669429779, 'precision': 0.5249367356300354, 'recall': 0.7234292030334473, 'iou': 0.43719732761383057}
Model saved with IoU: 0.4372
Epoch [3/10], Train Loss: 0.4111, Val Loss: 0.3945, Val Scores: {'f1': 0.6261287331581116, 'precision': 0.6299970149993896, 'recall': 0.6223076581954956, 'iou': 0.45574048161506653}
Model saved with IoU: 0.4557
Epoch [4/10], Train Loss: 0.3893, Val Loss: 0.3672, Val Scores: {'f1': 0.6541129946708679, 'precision': 0.618061363697052, 'recall': 0.6946308612823486, 'iou': 0.48600882291793823}
Model saved with IoU: 0.4860
Epoch [5/10], Train Loss: 0.3718, Val Loss: 0.3673, Val Scores: {'f1': 0.644312858581543, 'precision': 0.6208953261375427, 'recall': 0.